# 02 - Baseline Evaluation

Benchmarks the **untuned** Mistral 7B Instruct v0.3 on our function-calling test set. Establishes the "before" metrics that we'll compare against after fine-tuning.

**Key question**: How well does the base model handle enterprise function calling out of the box?

In [0]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes peft tqdm

In [0]:
dbutils.library.restartPython()

In [0]:
import json
import sys
import os

PROJECT_ROOT = "/Workspace/Users/alberto.lapedriza@kpmg.co.uk/mistral-7b-enterprise-function-calling"  # <-- UPDATE THIS
sys.path.insert(0, PROJECT_ROOT)

from src.schemas import TOOL_SCHEMAS, SYSTEM_PROMPT
from src.utils import load_jsonl, dataset_stats
from src.inference import load_base_model, run_inference
from src.evaluation import evaluate_results
from src.reporting import (
    overall_summary, breakdown_by_category, breakdown_by_tool,
    print_qualitative_examples,
)

In [0]:
DATA_DIR = f"{PROJECT_ROOT}/data"
test_data = load_jsonl(f"{DATA_DIR}/test.jsonl")

print(f"Test set size: {len(test_data)} examples")
dataset_stats(test_data)

In [0]:
model, tokenizer = load_base_model()

In [0]:
test_data = test_data[:3]
results = run_inference_on_test_set(model, tokenizer, test_data)

In [0]:
# Before computing metrics, eyeball a few raw predictions to make sure
# inference is working correctly.
for i in range(3):
    print(f"\n{'='*60}")
    print(f"Example {i+1} | Category: {results[i]['category']}")
    print(f"{'='*60}")
    print(f"USER: {results[i]['input_messages'][1]['content'][:200]}...")
    print(f"\nEXPECTED: {results[i]['expected'][:200]}...")
    print(f"\nPREDICTED: {results[i]['predicted'][:200]}...")

In [0]:
eval_df = evaluate_results(results)
print(f"Evaluation complete: {len(eval_df)} examples scored")
eval_df.head(10)

In [0]:
summary = overall_summary(eval_df)
summary

In [0]:
# Where does the model struggle most? We expect:
# - **Simple**: Decent — parameters are explicit
# - **Complex**: Weaker — requires inference from implicit phrasing
# - **Multi-tool**: Weakest — 7B models often fail at parallel tool calls
# - **Ambiguous**: Variable — tests fine-grained disambiguation
# - **No-tool**: Depends — does the model over-trigger function calls?

cat_breakdown = breakdown_by_category(eval_df)
cat_breakdown

In [0]:
# Are some tools harder than others? Tools with nested objects/arrays
# (e.g., `create_invoice`, `send_notification`) should be trickier.

tool_breakdown = breakdown_by_tool(eval_df, results)
tool_breakdown

In [0]:
# Save everything so we can compare against fine-tuned results in Notebook 04.

OUTPUT_DIR = f"{PROJECT_ROOT}/results"

# Save results to JSON
with open(f"{OUTPUT_DIR}/baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Save eval_df as CSV
eval_df.to_csv(f"{OUTPUT_DIR}/baseline_eval_df.csv", index=False)

# Save summary as CSV
summary.to_csv(f"{OUTPUT_DIR}/baseline_summary.csv", index=False)

# Save cat_breakdown as CSV
cat_breakdown.to_csv(f"{OUTPUT_DIR}/baseline_cat_breakdown.csv", index=False)

# Save tool_breakdown as CSV
tool_breakdown.to_csv(f"{OUTPUT_DIR}/baseline_tool_breakdown.csv", index=False)

In [0]:
# The numbers tell part of the story, but seeing actual outputs is
# essential for understanding *how* the model fails.

print_qualitative_examples(results, eval_df, n_per_bucket=3)

In [0]:
# Failure-mode breakdown: non-overlapping buckets so counts sum to total errors.

tool_examples = eval_df[eval_df["category"] != "no_tool"]
n_tool = len(tool_examples)

# Build a waterfall: each bucket removes examples claimed by previous ones
parseable = tool_examples[tool_examples["json_valid"] == True]
correct_func = parseable[parseable["func_name_correct"] == True]

failure_buckets = {
    "JSON Parse Failures": len(tool_examples) - len(parseable),
    "Wrong Function Name": len(parseable) - len(correct_func),
    "Missing Required Fields": (correct_func["required_fields"] < 1.0).sum() if len(correct_func) else 0,
    "Hallucinated Params": (correct_func["hallucinated_params"] > 0).sum() if len(correct_func) else 0,
}

print(f"Tool-call examples: {n_tool}\n")
for label, count in failure_buckets.items():
    print(f"  {label:.<30} {count:>4} / {n_tool}  ({count/n_tool:.1%})")

# No-tool restraint (separate population)
no_tool = eval_df[eval_df["category"] == "no_tool"]
if len(no_tool) > 0:
    false_triggers = (no_tool["no_tool_restraint"] == False).sum()
    print(f"\nNo-Tool False Triggers: {false_triggers} / {len(no_tool)} "
          f"({false_triggers/len(no_tool):.1%})")

In [0]:
# _(Fill this in after running — these become your README talking points)_
#
# **Add your observations here after seeing the results:**
#
# 1. **JSON Validity**: ...% — the base model [can/struggles to] produce valid JSON
# 2. **Function Selection**: ...% — the model [often/sometimes/rarely] picks the wrong tool
# 3. **Hardest Category**: [...] — as expected/surprisingly, because...
# 4. **Hardest Tool**: [...] — likely due to [nested objects/similar to another tool/many required params]
# 5. **No-Tool Restraint**: ...% — the model [does/doesn't] over-trigger function calls
# 6. **Most Common Failure Mode**: [JSON formatting / wrong function / missing fields / hallucinated params]
#
# **These weaknesses are exactly what fine-tuning should address.**

# %% [markdown]
# ## 14. Clean Up
#
# Free GPU memory before moving to the training notebook.

# %%
import gc
import torch

del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")